# 5. Kreiranje dimenzijskog modela (Star shema)

Ovaj notebook kreira tablice za star shemu u MySQL:
- `dim_projekt` — dimenzija projekata
- `dim_tehnicar` — dimenzija tehničara (reporteri i assigneei)
- `dim_prioritet` — razina prioriteta ticketa
- `dim_status` — status ticketa
- `dim_vrijeme` — vremenska dimenzija
- `fact_support_tickets` — tablica činjenica s metrikama

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'fipu_srp_projekt')

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD nije postavljen! Kreiraj .env datoteku.")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
print(f"Spojeno na bazu: {DB_NAME} ({DB_HOST})")

Spojeno na bazu: fipu_srp_projekt (localhost)


## 5.1 DDL — Kreiranje tablica

In [2]:
# Prvo dropaj stare tablice (redoslijed: fact -> dimenzije zbog FK)
drop_statements = [
    "DROP TABLE IF EXISTS fact_support_tickets;",
    "DROP TABLE IF EXISTS dim_vrijeme;",
    "DROP TABLE IF EXISTS dim_projekt;",
    "DROP TABLE IF EXISTS dim_tehnicar;",
    "DROP TABLE IF EXISTS dim_prioritet_status;",
    "DROP TABLE IF EXISTS dim_prioritet;",
    "DROP TABLE IF EXISTS dim_status;",
]

with engine.connect() as conn:
    trans = conn.begin()
    for stmt in drop_statements:
        conn.execute(text(stmt))
    trans.commit()
    print("Stare tablice obrisane.")

sql_statements = [
    """
    CREATE TABLE IF NOT EXISTS dim_projekt (
        projekt_key INT AUTO_INCREMENT PRIMARY KEY,
        naziv_projekta VARCHAR(255)
    );
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_tehnicar (
        tehnicar_key INT AUTO_INCREMENT PRIMARY KEY,
        ime_prezime VARCHAR(255)
    );
    """,
    """
    CREATE TABLE dim_prioritet (
        prioritet_key INT AUTO_INCREMENT PRIMARY KEY,
        razina_prioriteta VARCHAR(50)
    );
    """,
    """
    CREATE TABLE dim_status (
        status_key INT AUTO_INCREMENT PRIMARY KEY,
        naziv_statusa VARCHAR(50)
    );
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_vrijeme (
        vrijeme_key DATE PRIMARY KEY,
        dan INT,
        mjesec INT,
        godina INT,
        kvartal INT,
        dan_u_tjednu VARCHAR(20)
    );
    """,
    """
    CREATE TABLE IF NOT EXISTS fact_support_tickets (
        ticket_id INT PRIMARY KEY,
        projekt_key INT,
        reporter_key INT,
        assignee_key INT,
        prioritet_key INT,
        status_key INT,
        vrijeme_key DATE,
        vrijeme_rjesavanja_sati DECIMAL(10, 2),
        broj_komentara INT,
        sati_open DECIMAL(10, 2),
        sati_in_progress DECIMAL(10, 2),
        sati_resolved DECIMAL(10, 2),
        sati_waiting DECIMAL(10, 2),
        CONSTRAINT fk_projekt FOREIGN KEY (projekt_key) REFERENCES dim_projekt(projekt_key),
        CONSTRAINT fk_reporter FOREIGN KEY (reporter_key) REFERENCES dim_tehnicar(tehnicar_key),
        CONSTRAINT fk_assignee FOREIGN KEY (assignee_key) REFERENCES dim_tehnicar(tehnicar_key),
        CONSTRAINT fk_prioritet FOREIGN KEY (prioritet_key) REFERENCES dim_prioritet(prioritet_key),
        CONSTRAINT fk_status FOREIGN KEY (status_key) REFERENCES dim_status(status_key),
        CONSTRAINT fk_vrijeme FOREIGN KEY (vrijeme_key) REFERENCES dim_vrijeme(vrijeme_key)
    );
    """
]

try:
    with engine.connect() as conn:
        trans = conn.begin()
        for stmt in sql_statements:
            conn.execute(text(stmt))
        trans.commit()
        print("Sve tablice dimenzijskog modela su uspješno kreirane.")
except Exception as e:
    print(f"Greška: {e}")

Stare tablice obrisane.
Sve tablice dimenzijskog modela su uspješno kreirane.
